# Motion-Guided Structural Regularization Training

Use this Kaggle notebook after adding these as Kaggle input datasets:

1. `gloss_training_colab.zip` containing `gloss_structure/`
2. `data_iSign.zip` containing `data_iSign/iSign_v1.1.csv`, `data_iSign/norm_stats.npz`, and `data_iSign/poses/*.npy`

This notebook trains:

- Baseline iSign English CTC: `--struct_source none`
- Motion-guided structural model: `--struct_source motion`

In [ ]:
import os
import zipfile
import shutil
from pathlib import Path

WORK_DIR = Path('/kaggle/working/ISL-CorrectionPipeline')
INPUT_DIR = Path('/kaggle/input')

shutil.rmtree(WORK_DIR, ignore_errors=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

print('Kaggle input folders:')
for p in sorted(INPUT_DIR.iterdir()):
    print(' -', p)

In [ ]:
def find_file(name):
    matches = list(INPUT_DIR.rglob(name))
    if not matches:
        raise FileNotFoundError(f'Could not find {name} under /kaggle/input. Add it as a Kaggle dataset.')
    return matches[0]

PROJECT_ZIP = find_file('gloss_training_colab.zip')
ISIGN_ZIP = find_file('data_iSign.zip')

print('Project zip:', PROJECT_ZIP)
print('iSign zip:', ISIGN_ZIP)

with zipfile.ZipFile(PROJECT_ZIP, 'r') as z:
    z.extractall(WORK_DIR)

# Detect folder containing gloss_structure
phase2_files = list(WORK_DIR.rglob('train_phase2_isign.py'))
if not phase2_files:
    raise FileNotFoundError('train_phase2_isign.py not found after extracting project zip')

PROJECT_DIR = phase2_files[0].parents[1]
print('Using PROJECT_DIR:', PROJECT_DIR)

with zipfile.ZipFile(ISIGN_ZIP, 'r') as z:
    z.extractall(PROJECT_DIR)

os.chdir(PROJECT_DIR)
print('Current dir:', Path.cwd())

In [ ]:
# Fix possible nested data_iSign/data_iSign structure if your zip includes an extra parent folder.
expected_csv = PROJECT_DIR / 'data_iSign' / 'iSign_v1.1.csv'
if not expected_csv.exists():
    matches = list(PROJECT_DIR.rglob('iSign_v1.1.csv'))
    print('Found iSign CSV candidates:', matches)
    if matches:
        real_data_dir = matches[0].parent
        target_data_dir = PROJECT_DIR / 'data_iSign'
        if real_data_dir != target_data_dir:
            shutil.rmtree(target_data_dir, ignore_errors=True)
            shutil.copytree(real_data_dir, target_data_dir)

print('Phase2 script exists:', (PROJECT_DIR / 'gloss_structure' / 'train_phase2_isign.py').exists())
print('iSign CSV exists:', (PROJECT_DIR / 'data_iSign' / 'iSign_v1.1.csv').exists())
print('iSign poses exists:', (PROJECT_DIR / 'data_iSign' / 'poses').exists())
print('iSign norm exists:', (PROJECT_DIR / 'data_iSign' / 'norm_stats.npz').exists())
print('Pose files:', len(list((PROJECT_DIR / 'data_iSign' / 'poses').glob('*.npy'))) if (PROJECT_DIR / 'data_iSign' / 'poses').exists() else 0)

In [ ]:
!python -m gloss_structure.train_phase2_isign --help

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## Train Baseline: No Structural Loss

In [ ]:
!python -m gloss_structure.train_phase2_isign \
  --manifest data_iSign/iSign_v1.1.csv \
  --pose_dir data_iSign/poses \
  --label_column english \
  --norm_stats data_iSign/norm_stats.npz \
  --out_dir /kaggle/working/isign_baseline_no_struct \
  --english_token_mode char \
  --struct_source none \
  --lambda_struct 0.0 \
  --full_epochs 40 \
  --batch_size 16 \
  --lr 1e-4 \
  --hidden_size 128 \
  --num_layers 1 \
  --dropout 0.3 \
  --device cuda

## Train Motion-Guided Structural Model

In [ ]:
!python -m gloss_structure.train_phase2_isign \
  --manifest data_iSign/iSign_v1.1.csv \
  --pose_dir data_iSign/poses \
  --label_column english \
  --norm_stats data_iSign/norm_stats.npz \
  --out_dir /kaggle/working/isign_motion_struct_phase2 \
  --english_token_mode char \
  --struct_source motion \
  --lambda_struct 0.03 \
  --full_epochs 40 \
  --batch_size 16 \
  --lr 1e-4 \
  --hidden_size 128 \
  --num_layers 1 \
  --dropout 0.3 \
  --device cuda

## Compare Best Validation Token Error

In [ ]:
import json
from pathlib import Path

def best_val(history_path):
    history_path = Path(history_path)
    if not history_path.exists():
        return None
    with open(history_path, 'r') as f:
        hist = json.load(f)
    best = min(hist, key=lambda x: x['val']['token_er'])
    return {
        'stage': best['stage'],
        'epoch': best['epoch'],
        'token_er': best['val']['token_er'],
        'loss': best['val']['loss'],
        'struct': best['val']['struct'],
    }

baseline = best_val('/kaggle/working/isign_baseline_no_struct/history.json')
motion = best_val('/kaggle/working/isign_motion_struct_phase2/history.json')

print('Baseline:', baseline)
print('Motion:', motion)

if baseline and motion:
    delta = baseline['token_er'] - motion['token_er']
    print('Token ER improvement:', delta)

## Zip Outputs For Download

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/isign_training_outputs', 'zip', '/kaggle/working')
print('Created /kaggle/working/isign_training_outputs.zip')